In [2]:
# ===== Bloco 0: Imports, parâmetros e MLflow =====
import os
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler

# XGBoost (opcional)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# Keras / TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import mlflow
import mlflow.sklearn
import mlflow.tensorflow
from mlflow.tracking import MlflowClient

# ========= Parâmetros =========
DATA_PATH = Path("dados/online_shoppers_intention.csv")
MLRUNS_DIR = Path("mlruns").resolve()
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "batch_retraining"
REGISTRY_MODEL_NAME = "ecommerce_conversion_model"

WINDOW_MONTHS = 6       # janela rolling para treino
PROMOTE_TO_PRODUCTION = False  # cuidado: promove direto para Production se True
RANDOM_STATE = 42
N_JOBS = -1

# MLflow setup
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)
print("XGBoost disponível?", HAS_XGB)
print("TensorFlow:", tf.__version__)


Tracking URI: file:///D:/Drive/Academico/DataScience_XP/ProjetoAplicado/projetoAplicadoDSXP/mlruns
Experiment: batch_retraining
XGBoost disponível? True
TensorFlow: 2.17.0


In [3]:
# ===== Bloco 1: encode_features (igual ao projeto) =====
def encode_features(df_in):
    df = df_in.copy()

    # (a) Month -> ordinal (ordem temporal)
    meses_ordem = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_map = {m:i+1 for i,m in enumerate(meses_ordem)}  # Feb=1, Mar=2, ... Dec=10
    df['Month'] = df['Month'].map(month_map).astype('int64')

    # (b) VisitorType -> One-Hot (base = New)
    df['VisitorType'] = df['VisitorType'].replace({
        'Returning_Visitor': 'Returning',
        'New_Visitor': 'New',
        'Other': 'Other'
    })
    df = pd.get_dummies(df, columns=['VisitorType'], drop_first=True)
    # Cria colunas: VisitorType_Other, VisitorType_Returning (base implícita = New)

    # (c) Weekend -> garantir inteiro
    if df['Weekend'].dtype != 'int64' and df['Weekend'].dtype != 'int32':
        df['Weekend'] = df['Weekend'].astype(int)

    return df


In [4]:
# ===== Bloco 2: Utils =====
def last_k_months_split(df_enc, window_months=WINDOW_MONTHS):
    """
    Split temporal:
      - current_month = df_enc['Month'].max()
      - val_month = current_month - 1
      - treino = meses de (val_month - window_months + 1) ... val_month
    """
    current_month = int(df_enc['Month'].max())
    val_month = max(1, current_month - 1)
    min_train_month = max(1, val_month - window_months + 1)

    df_train = df_enc[(df_enc['Month'] >= min_train_month) & (df_enc['Month'] <= val_month)].copy()
    df_val   = df_enc[df_enc['Month'] == val_month].copy()

    return df_train, df_val, current_month, val_month, min_train_month

def make_Xy(df_enc):
    y = df_enc['Revenue'].astype(int).values
    X = df_enc.drop(columns=['Revenue']).copy()
    return X, y

def sweep_threshold(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.linspace(0.1, 0.9, 17)
    best = {"threshold": 0.5, "f1": -1, "precision": None, "recall": None}
    for t in grid:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best["f1"]:
            best["f1"] = f1
            best["threshold"] = float(t)
            best["precision"] = precision_score(y_true, y_pred, zero_division=0)
            best["recall"] = recall_score(y_true, y_pred, zero_division=0)
    return best

def evaluate_at_threshold(y_true, y_prob, t):
    y_pred = (y_prob >= t).astype(int)
    return {
        "acc":  accuracy_score(y_true, y_pred),
        "prec": precision_score(y_true, y_pred, zero_division=0),
        "rec":  recall_score(y_true, y_pred, zero_division=0),
        "f1":   f1_score(y_true, y_pred, zero_division=0),
        "auc":  roc_auc_score(y_true, y_prob),
    }


In [5]:
# ===== Bloco 3: Carregar, encodar e split =====
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH, encoding="utf-8")
df_enc = encode_features(df_raw)

if "Revenue" not in df_enc.columns:
    raise ValueError("Coluna 'Revenue' não encontrada após encode_features.")

df_train, df_val, current_m, val_m, min_train_m = last_k_months_split(df_enc, WINDOW_MONTHS)

print(f"Janela de treino: meses {min_train_m} .. {val_m} | mês atual={current_m}")
print(f"Treino: {df_train.shape} | Validação: {df_val.shape}")

X_train, y_train = make_Xy(df_train)
X_val, y_val     = make_Xy(df_val)
print("X_train:", X_train.shape, "| X_val:", X_val.shape)


Janela de treino: meses 4 .. 9 | mês atual=10
Treino: (5148, 19) | Validação: (2998, 19)
X_train: (5148, 18) | X_val: (2998, 18)


In [6]:
# ===== Bloco 4: Undersampling no treino =====
rus = RandomUnderSampler(random_state=RANDOM_STATE)
Xu, yu = rus.fit_resample(X_train, y_train)

print("Distribuição original treino:", np.bincount(y_train))
print("Distribuição após undersampling:", np.bincount(yu))


Distribuição original treino: [4016 1132]
Distribuição após undersampling: [1132 1132]


In [7]:
# ===== Bloco 5: Treinar Random Forest =====
param_dist_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [6, 8, 10, 12, None],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    class_weight="balanced_subsample"
)

rf_search = RandomizedSearchCV(
    rf, param_distributions=param_dist_rf, n_iter=20, cv=3,
    scoring="f1", random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=0
)
rf_search.fit(Xu, yu)
rf_best = rf_search.best_estimator_

# Avaliação na validação (threshold tuning)
rf_prob = rf_best.predict_proba(X_val)[:, 1]
rf_base = evaluate_at_threshold(y_val, rf_prob, 0.5)
rf_tune = sweep_threshold(y_val, rf_prob)

print("RF base:", rf_base)
print("RF best threshold:", rf_tune)


RF base: {'acc': 0.8655770513675783, 'prec': 0.6727976766698935, 'rec': 0.9144736842105263, 'f1': 0.7752370329057445, 'auc': 0.952498118620949}
RF best threshold: {'threshold': 0.55, 'f1': 0.780862374483166, 'precision': 0.7084673097534834, 'recall': 0.8697368421052631}


In [8]:
# ===== Bloco 6: Treinar XGBoost (se disponível) =====
if HAS_XGB:
    param_dist_xgb = {
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.03, 0.05, 0.1],
        "min_child_weight": [1, 3, 5],
        "gamma": [0, 0.1, 0.2],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [0.5, 1.0, 2.0],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "n_estimators": [300, 400, 600]
    }
    xgb = XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=N_JOBS,
        tree_method="hist"
    )
    xgb_search = RandomizedSearchCV(
        xgb, param_distributions=param_dist_xgb, n_iter=20, cv=3,
        scoring="f1", random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=0
    )
    xgb_search.fit(Xu, yu)
    xgb_best = xgb_search.best_estimator_

    xgb_prob = xgb_best.predict_proba(X_val)[:, 1]
    xgb_base = evaluate_at_threshold(y_val, xgb_prob, 0.5)
    xgb_tune = sweep_threshold(y_val, xgb_prob)

    print("XGB base:", xgb_base)
    print("XGB best threshold:", xgb_tune)
else:
    xgb_best = None
    xgb_prob = None
    xgb_tune = None


XGB base: {'acc': 0.8362241494329553, 'prec': 0.6287081339712919, 'rec': 0.8644736842105263, 'f1': 0.72797783933518, 'auc': 0.921229010864964}
XGB best threshold: {'threshold': 0.6, 'f1': 0.7388688327316485, 'precision': 0.6807095343680709, 'recall': 0.8078947368421052}


In [9]:
# ===== Bloco 7: Treinar MLP (Keras) =====
# Padronização apenas para MLP
scaler = StandardScaler()
Xu_mlp = scaler.fit_transform(Xu)
Xval_mlp = scaler.transform(X_val)

def build_mlp(input_dim, hidden=(128,), dropout=0.0, lr=1e-3):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for h in hidden:
        model.add(layers.Dense(h, activation="relu"))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

mlp = build_mlp(input_dim=Xu_mlp.shape[1], hidden=(128,), dropout=0.0, lr=1e-3)
es = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss")
mlp.fit(
    Xu_mlp, yu,
    validation_data=(Xval_mlp, y_val),
    epochs=40, batch_size=128, verbose=0,
    callbacks=[es]
)

mlp_prob = mlp.predict(Xval_mlp, verbose=0).ravel()
mlp_base = evaluate_at_threshold(y_val, mlp_prob, 0.5)
mlp_tune = sweep_threshold(y_val, mlp_prob)

print("MLP base:", mlp_base)
print("MLP best threshold:", mlp_tune)


MLP base: {'acc': 0.7788525683789193, 'prec': 0.5457979225684608, 'rec': 0.7605263157894737, 'f1': 0.6355140186915887, 'auc': 0.8417901321668783}
MLP best threshold: {'threshold': 0.5, 'f1': 0.6355140186915887, 'precision': 0.5457979225684608, 'recall': 0.7605263157894737}


In [10]:
# ===== Bloco 8: Log no MLflow =====
# Salvar ordem de colunas para produção
feature_cols = list(X_val.columns)
feat_path = Path("feature_cols.json")
with open(feat_path, "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

runs_summary = []

# RF
with mlflow.start_run(run_name="retrain__rf_undersample"):
    mlflow.set_tags({
        "phase": "batch_retraining",
        "encoding_fn": "encode_features_v1",
        "balance": "undersample",
        "window_months": WINDOW_MONTHS,
        "val_month": val_m,
        "current_month": current_m
    })
    mlflow.log_params({"model": "rf"})
    if hasattr(rf_search, "best_params_"):
        mlflow.log_params({f"rf__{k}": v for k, v in rf_search.best_params_.items()})

    # base (t=0.5) e tuned
    for k, v in evaluate_at_threshold(y_val, rf_prob, 0.5).items():
        mlflow.log_metric(f"val_base_{k}", float(v))
    for k, v in evaluate_at_threshold(y_val, rf_prob, rf_tune["threshold"]).items():
        mlflow.log_metric(f"val_tuned_{k}", float(v))

    mlflow.log_param("best_threshold", float(rf_tune["threshold"]))
    mlflow.sklearn.log_model(rf_best, artifact_path="model")
    mlflow.log_artifact(str(feat_path))

    runs_summary.append({
        "model": "rf_undersample",
        "run_id": mlflow.active_run().info.run_id,
        "best_threshold": float(rf_tune["threshold"]),
        "val_f1_tuned": float(evaluate_at_threshold(y_val, rf_prob, rf_tune["threshold"])["f1"])
    })

# XGB
if HAS_XGB and xgb_best is not None:
    with mlflow.start_run(run_name="retrain__xgb_undersample"):
        mlflow.set_tags({
            "phase": "batch_retraining",
            "encoding_fn": "encode_features_v1",
            "balance": "undersample",
            "window_months": WINDOW_MONTHS,
            "val_month": val_m,
            "current_month": current_m
        })
        mlflow.log_params({"model": "xgb"})
        if hasattr(xgb_search, "best_params_"):
            mlflow.log_params({f"xgb__{k}": v for k, v in xgb_search.best_params_.items()})

        for k, v in evaluate_at_threshold(y_val, xgb_prob, 0.5).items():
            mlflow.log_metric(f"val_base_{k}", float(v))
        for k, v in evaluate_at_threshold(y_val, xgb_prob, xgb_tune["threshold"]).items():
            mlflow.log_metric(f"val_tuned_{k}", float(v))

        mlflow.log_param("best_threshold", float(xgb_tune["threshold"]))
        mlflow.sklearn.log_model(xgb_best, artifact_path="model")
        mlflow.log_artifact(str(feat_path))

        runs_summary.append({
            "model": "xgb_undersample",
            "run_id": mlflow.active_run().info.run_id,
            "best_threshold": float(xgb_tune["threshold"]),
            "val_f1_tuned": float(evaluate_at_threshold(y_val, xgb_prob, xgb_tune["threshold"])["f1"])
        })

# MLP (Keras)
with mlflow.start_run(run_name="retrain__mlp_undersample"):
    mlflow.set_tags({
        "phase": "batch_retraining",
        "encoding_fn": "encode_features_v1",
        "balance": "undersample",
        "window_months": WINDOW_MONTHS,
        "val_month": val_m,
        "current_month": current_m
    })
    mlflow.log_params({"model": "mlp", "hidden": "(128,)", "lr": 1e-3, "epochs": 40, "batch_size": 128})

    for k, v in evaluate_at_threshold(y_val, mlp_prob, 0.5).items():
        mlflow.log_metric(f"val_base_{k}", float(v))
    for k, v in evaluate_at_threshold(y_val, mlp_prob, mlp_tune["threshold"]).items():
        mlflow.log_metric(f"val_tuned_{k}", float(v))

    mlflow.log_param("best_threshold", float(mlp_tune["threshold"]))

    # Log do modelo Keras
    mlflow.tensorflow.log_model(mlp, artifact_path="keras_model")

    # Log do scaler (mean/scale) para usar na produção
    np.save("scaler_mean.npy", scaler.mean_)
    np.save("scaler_scale.npy", scaler.scale_)
    mlflow.log_artifact("scaler_mean.npy")
    mlflow.log_artifact("scaler_scale.npy")
    os.remove("scaler_mean.npy"); os.remove("scaler_scale.npy")

    mlflow.log_artifact(str(feat_path))

    runs_summary.append({
        "model": "mlp_undersample",
        "run_id": mlflow.active_run().info.run_id,
        "best_threshold": float(mlp_tune["threshold"]),
        "val_f1_tuned": float(evaluate_at_threshold(y_val, mlp_prob, mlp_tune["threshold"])["f1"])
    })

# limpar arquivo temporário
feat_path.unlink(missing_ok=True)

print("\nRuns registrados no MLflow:")
for r in runs_summary:
    print(r)


2025/08/20 21:20:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/08/20 21:20:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/08/20 21:20:26 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2025/08/20 21:20:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Runs registrados no MLflow:
{'model': 'rf_undersample', 'run_id': '32ccfdbd7c12456d915c856642fc31b1', 'best_threshold': 0.55, 'val_f1_tuned': 0.780862374483166}
{'model': 'xgb_undersample', 'run_id': '51e3853b27a447e290b6ef6cf3341a63', 'best_threshold': 0.6, 'val_f1_tuned': 0.7388688327316485}
{'model': 'mlp_undersample', 'run_id': 'b3cd07cbf66841b2ae5369298663a4c7', 'best_threshold': 0.5, 'val_f1_tuned': 0.6355140186915887}


In [11]:
# ===== Bloco 9: Registrar campeão no Model Registry =====
# Escolhe por maior val_f1_tuned
champion = max(runs_summary, key=lambda d: d["val_f1_tuned"])
print("\n=== Campeão (val_f1_tuned) ===")
print(champion)

# Registrar modelo do run vencedor
winner_model_uri = f"runs:/{champion['run_id']}/model"
registered = mlflow.register_model(model_uri=winner_model_uri, name=REGISTRY_MODEL_NAME)
print(f"Registrado no Model Registry: name={REGISTRY_MODEL_NAME}, version={registered.version}")

# Mover para Staging
client.transition_model_version_stage(
    name=REGISTRY_MODEL_NAME,
    version=registered.version,
    stage="Staging",
    archive_existing_versions=False
)
print(f"Disponível em Staging: {REGISTRY_MODEL_NAME} v{registered.version}")

# (Opcional) promover direto para Production
if PROMOTE_TO_PRODUCTION:
    client.transition_model_version_stage(
        name=REGISTRY_MODEL_NAME,
        version=registered.version,
        stage="Production",
        archive_existing_versions=True
    )
    print(f"Promovido para Production: {REGISTRY_MODEL_NAME} v{registered.version}")



=== Campeão (val_f1_tuned) ===
{'model': 'rf_undersample', 'run_id': '32ccfdbd7c12456d915c856642fc31b1', 'best_threshold': 0.55, 'val_f1_tuned': 0.780862374483166}
Registrado no Model Registry: name=ecommerce_conversion_model, version=1
Disponível em Staging: ecommerce_conversion_model v1


Successfully registered model 'ecommerce_conversion_model'.
Created version '1' of model 'ecommerce_conversion_model'.
